# sync

> Synced navigation between two card stacks — source stack navigation drives target stack to same index

In [ ]:
#| default_exp js.sync

In [ ]:
#| export
from typing import Optional

## Sync JS Generator

Generates a standalone JS snippet that syncs a target card stack to the source
card stack's focused index. When enabled, any navigation in the source stack
triggers `htmx.ajax('POST', targetUrl, ...)` to navigate the target stack to
the same index.

The sync uses an `htmx:afterSettle` handler that reads the source stack's
hidden `focused_index` input by ID on each event. This deferred lookup works
even if the source stack hasn't initialized when the JS first runs.

Out-of-range indices are clamped server-side by `nav_to_index`.

### Listener cleanup

Uses a keyed `window` reference pattern to remove previous handlers on re-render,
preventing accumulation across step navigation.

In [ ]:
#| export
def generate_card_stack_sync_js(
    source_input_id:str,  # ID of source stack's focused_index hidden input
    target_nav_url:str,  # URL for target stack's nav_to_index route
    toggle_fn_name:str="toggleSyncedNav",  # Name for window toggle function
    sync_key:str="_cardStackSync",  # Window key for state + cleanup
) -> str:  # Standalone JS snippet (not inside any IIFE)
    """Generate JS for synced navigation between two card stacks.
    
    Source stack navigation drives target stack to the same focused index.
    Toggle on/off via window[toggle_fn_name]() or toolbar button.
    
    The settle handler and source input lookup are deferred — they work
    even if the source stack hasn't initialized when this JS first runs.
    Out-of-range indices are clamped server-side by nav_to_index.
    """
    return f"""
    (function() {{
        // Clean up previous settle handler if re-rendered
        var settleKey = '{sync_key}_settle';
        if (window[settleKey]) {{
            document.body.removeEventListener('htmx:afterSettle', window[settleKey]);
        }}

        // Sync state
        var state = window.{sync_key} = {{
            enabled: false,
            targetUrl: '{target_nav_url}',
            lastSyncedIndex: -1,
        }};

        // Toggle function (called by KeyAction or toolbar button)
        window.{toggle_fn_name} = function() {{
            state.enabled = !state.enabled;
            return state.enabled;
        }};

        // Sync function — navigate target to source's index
        function syncTarget(sourceIndex) {{
            if (!state.enabled) return;
            var idx = parseInt(sourceIndex, 10);
            if (isNaN(idx) || idx < 0) return;
            if (idx === state.lastSyncedIndex) return;
            state.lastSyncedIndex = idx;
            htmx.ajax('POST', state.targetUrl, {{
                values: {{ target_index: idx }},
                swap: 'none',
            }});
        }}

        // Settle handler — reads source input by ID on each event
        // (deferred lookup works even if source stack initializes after this JS runs)
        var settleHandler = function(evt) {{
            if (!state.enabled) return;
            var input = document.getElementById('{source_input_id}');
            if (input) syncTarget(input.value);
        }};
        window[settleKey] = settleHandler;
        document.body.addEventListener('htmx:afterSettle', settleHandler);
    }})();
    """

## Tests

In [ ]:
# Test basic generation
js = generate_card_stack_sync_js(
    source_input_id="seg-focused-index",
    target_nav_url="/align/nav_to_index",
    toggle_fn_name="toggleSyncedNav",
    sync_key="_segAlignSync",
)

# Verify key elements are present
assert 'toggleSyncedNav' in js
assert '_segAlignSync' in js
assert 'seg-focused-index' in js
assert '/align/nav_to_index' in js
assert 'htmx.ajax' in js
assert 'state.enabled' in js
assert 'afterSettle' in js
assert 'lastSyncedIndex' in js  # Dedup guard
assert 'removeEventListener' in js  # Cleanup

# Verify settle handler is registered unconditionally (no sourceInput gate)
assert 'document.body.addEventListener' in js

print('Sync JS generator tests passed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()